<h1 style="text-align:center; 
           background:rgb(139, 217, 144); 
           color: rgb(4, 82, 9);
           font-size:45px; 
           padding:20px;
           border-radius:12px;">
     <b> Future Stock Prediction(ML)</b>
</h1>

### <b>🧑‍💼 Project Author</b>

***Author: Muhammad Usman***

***Artificial Intelligence/Machine Learning/DataEngineering Enthusiast***

***Date: MAY 2026***

<h2 style="text-align:center; 
           background: rgb(139, 217, 144);  
           color:rgb(4, 82, 9); 
           font-size:42px; 
           padding:12px;
           border-radius:12px;
           display: inline-block">
     <b>Preparing Data for Machine Learning</b>
</h2>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.offline as pyo

import sklearn as sk
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')


In [2]:
# Load dataset with Date column
appl_ml = pd.read_csv(
    "aapl_preprocessed.csv",
    parse_dates=['Date'],
    index_col='Date'
)

# Sort dates
appl_ml = appl_ml.sort_index()

In [3]:
appl_ml.head()

,Close,High,Low,Open,Volume,Year,Month,Day,IsYearStart,IsYearEnd,Returns,Returns (%),MA20,MA50,Volatility,lag1,lag2
Date,,,,,,,,,,,,,,,,,
2010-03-16,6.725198,6.741079,6.667070,6.717108,446908000,2010,March,Tuesday,0,0,0.002725,0.272533,6.346270,6.201066,0.012642,6.706920,6.789618
2010-03-17,6.715309,6.785122,6.689840,6.738680,450956800,2010,March,Wednesday,0,0,-0.001471,-0.147053,6.378585,6.207124,0.012552,6.725198,6.706920
2010-03-18,6.731189,6.741676,6.670064,6.714708,342109600,2010,March,Thursday,0,0,0.002365,0.236476,6.411125,6.213279,0.012546,6.715309,6.725198
2010-03-19,6.659277,6.748866,6.628714,6.735383,559445600,2010,March,Friday,0,0,-0.010683,-1.068337,6.441957,6.220038,0.012797,6.731189,6.715309
2010-03-22,6.734185,6.771639,6.596355,6.605944,456419600,2010,March,Monday,0,0,0.011249,1.124871,6.478406,6.228530,0.012590,6.659277,6.731189


In [5]:
features = ['Open', 'High', 'Low', 'Volume', 'MA20', 'MA50', 'Volatility', 'lag1', 'lag2']
target = 'Close'

X = appl_ml[features]
y = appl_ml[target]

In [6]:
split = int(len(appl_ml) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

### **Observation**
---
- Data is split chronologically (not randomly) to preserve time order.
- This mimics real-world forecasting conditions.

<h2 style="text-align:center; 
           background: rgb(139, 217, 144);  
           color:rgb(4, 82, 9); 
           font-size:42px; 
           padding:12px;
           border-radius:12px;
           display: inline-block">
     <b>Model Training</b>
</h2>

In [7]:
appl_ml.isnull().sum()

Close          0
High           0
Low            0
Open           0
Volume         0
Year           0
Month          0
Day            0
IsYearStart    0
IsYearEnd      0
Returns        0
Returns (%)    0
MA20           0
MA50           0
Volatility     0
lag1           0
lag2           0
dtype: int64

In [8]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_preds = lr_model.predict(X_test)

In [9]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)

**Evaluation**

In [57]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    return mae, rmse

# Linear Regression
lr_mae, lr_rmse = evaluate(y_test, lr_preds)
print(f"Linear Regression - MAE: {lr_mae:.2f}, RMSE: {lr_rmse:.2f}")

# Random Forest
rf_mae, rf_rmse = evaluate(y_test, rf_preds)
print(f"Random Forest - MAE: {rf_mae:.2f}, RMSE: {rf_rmse:.2f}")


Linear Regression - MAE: 0.91, RMSE: 1.24
Random Forest - MAE: 38.13, RMSE: 50.46


### **Observations**
---
The stock features had strong linear relationships with the target closing price. 

Linear Regression captured these relationships effectively, while Random Forest struggled with sequential time-series behavior and produced higher prediction errors

In [41]:
fig = go.Figure()

# Actual values
fig.add_trace(go.Scatter(
    y=y_test.values,
    mode='lines',
    name='Actual Price'
))

# LR predictions
fig.add_trace(go.Scatter(
    y=lr_preds,
    mode='lines',
    name='Predicted Price (LR)'
))

fig.update_layout(
    title="Actual vs Predicted Prices",
    xaxis_title="Time",
    yaxis_title="Price"
)

fig.show()

<h2 style="text-align:center; 
           background: rgb(139, 217, 144);  
           color:rgb(4, 82, 9); 
           font-size:42px; 
           padding:12px;
           border-radius:12px;
           display: inline-block">
     <b>Predictions</b>
</h2>

### **Generating Future Predictions**

In [11]:
# Number of future days
future_days = 7

# Last known feature row
last_input = X.iloc[-1].values.reshape(1, -1)

future_predictions = []

# Predict future values iteratively
for i in range(future_days):

    # Predict next close price
    pred = lr_model.predict(last_input)[0]

    # Store prediction
    future_predictions.append(pred)

    # Update input with predicted value
    last_input = np.roll(last_input, -1)

    # Put prediction in last position
    last_input[0, -1] = pred

### **Fixing Date Issue**

In [12]:
# Generate next 7 business days
future_dates = pd.date_range(
    start=appl_ml.index[-1] + pd.offsets.BDay(1),
    periods=future_days,
    freq='B'
)

### **Prediction Table**

In [13]:
future_df = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Close": future_predictions
})

# Add weekday names
future_df['Day'] = future_df['Date'].dt.day_name()

future_df

,Date,Predicted_Close,Day
0,2026-05-11,2.942447e+02,Monday
1,2026-05-12,4.264921e+07,Tuesday
2,2026-05-13,4.607574e+07,Wednesday
3,2026-05-14,-3.528107e+07,Thursday
4,2026-05-15,-3.238842e+07,Friday
5,2026-05-18,-2.426973e+07,Monday
6,2026-05-19,2.647157e+07,Tuesday


###  **OBSERVATIONS**
---
- The Linear Regression model performed well during evaluation on historical test data.
- However, during iterative future forecasting, the predicted values became unrealistic and unstable.
- Some predictions reached extremely large positive and negative values, which are not practically possible for stock prices.
- This issue occurs because Linear Regression does not naturally model sequential dependencies in time-series forecasting

### **Viz Through Graph**

In [14]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=future_df['Date'],
    y=future_df['Predicted_Close'],
    mode='lines+markers',
    name='Predicted Future Price',
    line=dict(color='green', width=3),
    marker=dict(size=8),
    hovertemplate=
        'Date: %{x}<br>' +
        'Predicted Close: %{y:.2f}<br>' +
        'Day: %{text}<extra></extra>',
    text=future_df['Day']
))

fig.update_layout(
    title="Future Stock Price Prediction (Next 7 Days)",
    xaxis_title="Date",
    yaxis_title="Predicted Close Price",
    hovermode="x"
)

fig.show()

<h2 style="text-align:center; 
           background: rgb(139, 217, 144);  
           color:rgb(4, 82, 9); 
           font-size:42px; 
           padding:12px;
           border-radius:12px;
           display: inline-block">
     <b>LSTM MODEL</b>
</h2>

---
#### ***🧠 Deep Learning Model: LSTM***

Traditional ML models cannot effectively remember `long-term sequential dependencies` in stock prices.

Therefore, `LSTM (Long Short-Term Memory)` was introduced for time-series forecasting.


### **Scaling Data**

In [15]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

scaled_close = scaler.fit_transform(
    appl_ml[['Close']]
)

In [16]:
scaler

,"feature_range feature_range: tuple (min, max), default=(0, 1)Desired range of transformed data.","(0, ...)"
,"copy copy: bool, default=TrueSet to False to perform inplace row normalization and avoid acopy (if the input is already a numpy array).",True
,"clip clip: bool, default=FalseSet to True to clip transformed values of held-out data toprovided `feature_range`.Since this parameter will clip values, `inverse_transform` may notbe able to restore the original data... note:: Setting `clip=True` does not prevent feature drift (a distribution shift between training and test data). The transformed values are clipped to the `feature_range`, which helps avoid unintended behavior in models sensitive to out-of-range inputs (e.g. linear models). Use with care, as clipping can distort the distribution of test data... versionadded:: 0.24",False


### **Create Sequences**

In [17]:
X_lstm = []
y_lstm = []

window_size = 60

for i in range(window_size, len(scaled_close)):
    X_lstm.append(scaled_close[i-window_size:i])
    y_lstm.append(scaled_close[i])

X_lstm = np.array(X_lstm)
y_lstm = np.array(y_lstm)

## **Train-Test Split**

In [18]:
split = int(len(X_lstm) * 0.8)

X_train_lstm = X_lstm[:split]
X_test_lstm = X_lstm[split:]

y_train_lstm = y_lstm[:split]
y_test_lstm = y_lstm[split:]

### **Build LSTM Model**

In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

model = Sequential()

model.add(LSTM(
    50,
    return_sequences=True,
    input_shape=(X_train_lstm.shape[1], 1)
))

model.add(Dropout(0.2))

model.add(LSTM(50))

model.add(Dropout(0.2))

model.add(Dense(1))

model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

## **Train Model**

In [21]:
history = model.fit(
    X_train_lstm,
    y_train_lstm,
    epochs=10,
    batch_size=32
)

Epoch 1/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - loss: 0.0021
Epoch 2/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - loss: 5.3206e-04
Epoch 3/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - loss: 6.2451e-04
Epoch 4/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step - loss: 5.9843e-04
Epoch 5/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - loss: 4.4649e-04
Epoch 6/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - loss: 4.3263e-04
Epoch 7/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - loss: 4.5723e-04
Epoch 8/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 4s 34ms/step - loss: 4.5317e-04
Epoch 9/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step - loss: 4.2727e-04
Epoch 10/10
101/101 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - loss: 4.0958e-04


## **Predict**

In [42]:
lstm_preds = model.predict(X_test_lstm)

# Convert back to real prices
lstm_preds = scaler.inverse_transform(lstm_preds)


# Actual values
y_test_actual = scaler.inverse_transform(
    y_test_lstm.reshape(-1,1)
)

26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step


In [23]:
lstm_preds

array([[146.29007],
       [145.63503],
       [145.00668],
       [144.74474],
       [144.90532],
       [145.19722],
       [145.6326 ],
       [145.991  ],
       [146.14828],
       [146.28067],
       [146.51247],
       [146.81737],
       [147.3229 ],
       [147.87311],
       [148.56134],
       [149.40424],
       [150.2031 ],
       [150.98674],
       [151.78531],
       [152.4206 ],
       [152.8711 ],
       [153.35913],
       [153.93323],
       [154.6794 ],
       [155.56416],
       [156.44778],
       [157.15942],
       [157.76625],
       [158.10432],
       [158.17409],
       [158.02419],
       [158.07213],
       [158.22255],
       [158.43568],
       [158.75125],
       [159.18115],
       [159.59679],
       [159.87784],
       [160.07144],
       [160.10019],
       [160.02103],
       [160.15704],
       [160.49754],
       [160.94461],
       [161.36858],
       [161.68347],
       [161.80602],
       [162.26372],
       [162.89706],
       [163.50143],


**lstm evaluation**

In [46]:
lstm_mae = mean_absolute_error(
    y_test_actual,
    lstm_preds
)

lstm_rmse = np.sqrt(
    mean_squared_error(
        y_test_actual,
        lstm_preds
    )
)

print("LSTM Performance")
print("MAE :", lstm_mae)
print("RMSE:", lstm_rmse)

LSTM Performance
MAE : 10.302370469073082
RMSE: 12.544117562659679


**Viz (LSTM vs Predicted values)**

In [47]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    y=y_test_actual.flatten(),
    mode='lines',
    name='Actual Price'
))

fig.add_trace(go.Scatter(
    y=lstm_preds.flatten(),
    mode='lines',
    name='Predicted Price (LSTM)'
))

fig.update_layout(
    title="LSTM Actual vs Predicted Prices",
    xaxis_title="Time",
    yaxis_title="Stock Price"
)

fig.show()

**Future forcasting using LSTM**

In [48]:
future_days = 7

last_sequence = scaled_close[-60:]

future_predictions = []

current_sequence = last_sequence.copy()

for _ in range(future_days):

    pred = model.predict(
        current_sequence.reshape(1,60,1),
        verbose=0
    )

    future_predictions.append(pred[0,0])

    current_sequence = np.append(
        current_sequence[1:],
        pred
    )

**converting future values**

In [49]:
future_predictions = scaler.inverse_transform(
    np.array(future_predictions).reshape(-1,1)
)

**future dates**

In [50]:
future_dates = pd.date_range(
    start=appl_ml.index[-1] + pd.offsets.BDay(1),
    periods=7,
    freq='B'
)

**final forcasting table**

In [51]:
future_df_lstm = pd.DataFrame({
    "Date": future_dates,
    "Predicted_Close": future_predictions.flatten()
})

future_df_lstm['Day'] = future_df_lstm['Date'].dt.day_name()

future_df_lstm

,Date,Predicted_Close,Day
0,2026-05-11,260.413513,Monday
1,2026-05-12,260.424652,Tuesday
2,2026-05-13,259.304657,Wednesday
3,2026-05-14,257.470215,Thursday
4,2026-05-15,255.211182,Friday
5,2026-05-18,252.720749,Monday
6,2026-05-19,250.124359,Tuesday


### **comparision evaluation table**

In [59]:
comparison_df = pd.DataFrame({

    "Model": [
        "Linear Regression",
        "Random Forest",
        "LSTM"
    ],

    "MAE": [
        lr_mae,
        rf_mae,
        lstm_mae
    ],

    "RMSE": [
        lr_rmse,
        rf_rmse,
        lstm_rmse
    ]
})

comparison_df

,Model,MAE,RMSE
0,Linear Regression,0.905245,1.243372
1,Random Forest,38.134439,50.457509
2,LSTM,10.302370,12.544118


### **Future Forecast Graph**

In [52]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=future_df_lstm['Date'],
    y=future_df_lstm['Predicted_Close'],
    mode='lines+markers',
    name='Future Forecast'
))

fig.update_layout(
    title="Next 7 Days Forecast using LSTM",
    xaxis_title="Date",
    yaxis_title="Predicted Price"
)

fig.show()

### **Observation**
---
- according to the above graph of 7 days forcast, the stock price is expected to remain same on 12th may, 2026 and then it will start to descrease in business days.

- Linear Regression achieved the best historical regression accuracy.
- Random Forest struggled with sequential market behavior.
- LSTM produced more stable future forecasting results.
- Deep learning models are more suitable for sequential stock forecasting tasks.

### **Exporting predicted values**

In [63]:
future_df_lstm.to_csv('future_predictions_lstm.csv', index=False)